# AgERA5 — download cache for BloomBench

Populate the local AgERA5 HDF5 store for every dataset in [`bloombench.config`](../src/pysephone/benchmarks/bloombench/config.py). Slow (queue-bound on CDS) but **resumable** — re-running skips entries already on disk.

Flow:

1. **Setup** — import shared config so the same dataset list as BloomBench is used.
2. **Parameters** — pick AgERA5 variables, tile size, and optional dataset subset.
3. **Plan** — load all selected datasets, merge their observations into one entry set, and report how many CDS requests will be needed. The merge step is the win: a tile shared by PEP725_Apple and PEP725_Pear becomes **one** CDS request instead of two, because bundling deduplicates across sources.
4. **Download** — single merged `get_agera5_data` call. Per-tile CDS queue waits dominate (~30s to ~10min each).
5. **Verify** — list the HDF5 store files and their sizes to confirm the cache is populated.

**Prerequisites:**
- CDS credentials at `~/.cdsapirc` (see https://cds.climate.copernicus.eu/api-how-to).
- Each licence accepted via the CDS web UI (https://cds.climate.copernicus.eu/datasets/sis-agrometeorological-indicators — scroll to *Terms of use*).


## 1. Setup — import shared config

In [ ]:
from __future__ import annotations

import time

import pandas as pd

from pysephone.benchmarks.bloombench import config as bc
from pysephone.data.agera5.download import (
    DEFAULT_TILE_DEG,
    build_entries,
    estimate_requests,
    get_agera5_data,
)
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.agera5 import AgEra5Features
from pysephone.dataset.util.calendar import Calendar

print(f'Datasets requested: {len(bc.DATASETS_REQUESTED)}')
print(f'Default tile size:  {DEFAULT_TILE_DEG}°')

## 2. Parameters

Edit these to control what gets fetched. For a first run, set `DATASETS_SUBSET` to a small dataset (e.g. `['GMU_Cherry_Switzerland']`) to confirm credentials/licence/tile-size all work end-to-end before kicking off the full BloomBench download.

In [ ]:
# AgERA5 variables to fetch (use AgERA5's NetCDF variable names).
# See _VARIABLE_SPECS in pysephone/data/agera5/download.py for the supported list.
DATA_KEYS = ['Temperature_Air_2m_Mean_24h']

# Tile size (degrees) for chunking CDS requests. Shrink if a region's per-request
# cost exceeds CDS's 400-unit limit.
TILE_DEG = DEFAULT_TILE_DEG  # 3.0°

# Subset filter — leave empty for all bench_config datasets, or list a few names
# (e.g. ['GMU_Cherry_Switzerland']) for a smoke test before the full BloomBench run.
DATASETS_SUBSET: list[str] = []

# Download policy:
#   None      — fetch only entries missing from the HDF5 store (recommended)
#   'forced'  — re-fetch everything, overwriting cached values
#   'skip'    — no network calls; useful for dry-runs
DOWNLOAD_MODE = None

# Calendar settings (must match bench_config so cached values stay usable).
SEASON_START  = bc.SEASON_START
SEASON_LENGTH = bc.SEASON_LENGTH

print('Variables:    ', DATA_KEYS)
print('Tile size:    ', f'{TILE_DEG}°')
print('Subset:       ', DATASETS_SUBSET or '(all)')
print('Download mode:', DOWNLOAD_MODE or '(missing only)')
print('Season:       ', f'starts {SEASON_START}, length {SEASON_LENGTH} days')

## 3. Plan — merge observations, count CDS requests

Loads every selected dataset, builds the union of required AgERA5 entries, and reports the work estimate. **No CDS calls.** Re-run this cell freely; it just reads from the local HDF5 store to count what's missing.

Bundling at this combined level is what saves requests across overlapping species (the 7 PEP725 species sharing a station set become one set of tile requests, not seven).

In [ ]:
datasets_to_run = [
    (name, target) for name, target in bc.DATASETS_REQUESTED
    if not DATASETS_SUBSET or name in DATASETS_SUBSET
]
print(f'Selected datasets: {len(datasets_to_run)}')

all_entries: set = set()
per_dataset_rows = []
for ds_name, _target in datasets_to_run:
    cal = Calendar(default_start=SEASON_START, default_length=SEASON_LENGTH)
    # debug_mode=True: we only need observations.iter_index() and coords —
    # no HDF5 store needed for entry-building.
    feats_dummy = AgEra5Features(calendar=cal, data_keys=DATA_KEYS, debug_mode=True)
    ds = Dataset.load(ds_name, calendar=cal, feature_providers=[feats_dummy])
    before = len(all_entries)
    ds_entries = build_entries(ds.observations, cal, DATA_KEYS)
    all_entries |= ds_entries
    added = len(all_entries) - before
    per_dataset_rows.append(dict(
        dataset=ds_name,
        entries=len(ds_entries),
        unique_added=added,
    ))

per_dataset_df = pd.DataFrame(per_dataset_rows)
print()
print(per_dataset_df.to_string(index=False))

# Estimate (queries the on-disk store, no CDS calls)
est = estimate_requests(all_entries, tile_deg=TILE_DEG)
print()
print(f'Total unique entries across all datasets: {est["total_entries"]:>8,}')
print(f'Already cached on disk:                  {est["total_entries"] - est["missing_entries"]:>8,}')
print(f'Missing — needs CDS:                     {est["missing_entries"]:>8,}')
print(f'CDS requests required (tile bundles):    {est["num_bundles"]:>8,}')

## 4. Download — one merged call

Single `get_agera5_data` call covering every dataset's missing entries. Cached entries are skipped automatically; restarting the kernel and re-running picks up where it left off (the plan cell shows the new estimate).

tqdm progress bars from `cdsapi` and pysephone will print inline; per-tile CDS queue waits range from ~30s to ~10min depending on load.

In [ ]:
t0 = time.time()
get_agera5_data(
    entries=all_entries,
    verbose=True,
    download_mode=DOWNLOAD_MODE,
    tile_deg=TILE_DEG,
)
elapsed = time.time() - t0
print(f'\nTotal wall time: {elapsed/3600:.2f} h  ({elapsed:.0f} s)')

## 5. Verify cache state

List the per-variable HDF5 stores on disk. After a successful run, expect one `.h5` file per variable in `DATA_KEYS`, each containing nested groups for every `(src_key, loc_id, year)` triple.

In [ ]:
from pysephone.data.agera5.download import _get_agera5_dir
from pysephone.paths import get_data_root

store_dir = _get_agera5_dir(get_data_root())
print(f'Store directory: {store_dir}')
print()
h5_files = sorted(store_dir.glob('*.h5'))
if not h5_files:
    print('  (no .h5 files yet — run the download cell first)')
for h5_path in h5_files:
    size_mb = h5_path.stat().st_size / 1024 / 1024
    print(f'  {h5_path.name:60s}  {size_mb:8.1f} MB')

# Re-run the estimate to confirm how many entries are now cached.
if h5_files:
    est_after = estimate_requests(all_entries, tile_deg=TILE_DEG)
    print()
    print(f'After download: cached {est_after["total_entries"] - est_after["missing_entries"]:,} '
          f'/ {est_after["total_entries"]:,} entries  '
          f'({est_after["num_bundles"]:,} CDS requests still pending)')